In [38]:
import json
from pathlib import Path
import pandas as pd

In [1]:
#Parameters

CANMOVES_SCALE_DEFINITION_PATH = '/content/canmovesScaleDefinition.json'
CANMOVES_RUN_SPEC_CONFIG_PATH = '/content/canmovesRunSpecConfig.json'
Fleet_CSV_PATH = '/content/Age Distribution.csv'
FleetAge_CSV_PATH = '/content/Fuel Distribution.csv'
EMME_Path = '/content/EMME.csv'
OUTPUT_EXCEL_PATH = '/content/output.xlsx'

#Functions

In [44]:
def read_json_file(json_path):
    """
    Read one JSON file and return its content as a dictionary.
    """

    json_path = Path(json_path)

    with open(json_path, "r", encoding="utf-8") as file:
        data = json.load(file)

    return data


def read_canmoves_json_files(scale_definition_path, run_spec_config_path):
    """
    Read the two CANMOVES JSON input files.
    """

    scale_definition = read_json_file(scale_definition_path)
    run_spec_config = read_json_file(run_spec_config_path)

    json_inputs = {
        "scale_definition": scale_definition,
        "run_spec_config": run_spec_config
    }

    return json_inputs

In [45]:
def read_csv_file(csv_path):
    """
    Read one CSV file and return it as a pandas DataFrame.
    """

    csv_path = Path(csv_path)

    df = pd.read_csv(csv_path, header=None)

    return df


def read_canmoves_csv_files(csv_paths):
    """
    Read multiple CANMOVES CSV input files.
    """

    csv_inputs = {}

    for csv_name, csv_path in csv_paths.items():
        csv_inputs[csv_name] = read_csv_file(csv_path)

    return csv_inputs

In [46]:
def generate_traffic_sheet(scale_definition, run_spec_config, EMME_Path):
    """
    Generate the Traffic sheet dataframe.
    """

    month_names = [
        "January", "February", "March", "April", "May", "June",
        "July", "August", "September", "October", "November", "December"
    ]

    domain_info = run_spec_config["domainInfo"]

    fleet_classification = scale_definition["fleetClassification"]

    if fleet_classification == "basic5":
        m6_fleet_flag = 0
    else:
        m6_fleet_flag = 1

    selected_month_index = int(domain_info["selectedMonth"])
    selected_month_name = month_names[selected_month_index]

    traffic_df = pd.DataFrame({
        0: [
            "EMME CSV file path",
            "M6 fleet flag",
            "Year",
            "Month",
            "Mean Temp",
            "Atmospheric Pressure"
        ],
        1: [
            EMME_Path,
            m6_fleet_flag,
            domain_info["selectedYear"],
            selected_month_name,
            domain_info["temp"],
            domain_info["atmosphericPressure"]
        ]
    })

    return traffic_df

In [47]:
def generate_region_sheet(run_spec_config):
    """
    Generate the Region sheet dataframe.
    """

    month_names = [
        "January", "February", "March", "April", "May", "June",
        "July", "August", "September", "October", "November", "December"
    ]

    domain_info = run_spec_config["domainInfo"]

    region_name = domain_info["regionName"]
    vdf_zone_id = domain_info["vdfZoneId"]
    temp = float(domain_info["temp"])
    atmospheric_pressure = float(domain_info["atmosphericPressure"])

    # Create empty dataframe with 6 rows and 14 columns: A to N
    region_df = pd.DataFrame("", index=range(6), columns=range(14))

    # Row 1
    region_df.iloc[0, 2:14] = month_names

    # Row 2
    region_df.iloc[1, 1] = 1

    # Row 3
    region_df.iloc[2, 1] = region_name
    region_df.iloc[2, 3] = "VDF ID"
    region_df.iloc[2, 4] = vdf_zone_id
    region_df.iloc[2, 5] = "Atm. P"
    region_df.iloc[2, 8] = atmospheric_pressure

    # Row 4: Tavg
    region_df.iloc[3, 1] = "Tavg"
    region_df.iloc[3, 2:14] = [temp] * 12

    # Row 5: Tmin
    region_df.iloc[4, 1] = "Tmin"
    region_df.iloc[4, 2:14] = [temp - 0.1] * 12

    # Row 6: Tmax
    region_df.iloc[5, 1] = "Tmax"
    region_df.iloc[5, 2:14] = [temp + 0.1] * 12

    return region_df

In [48]:
def generate_fleet_sheet(Fleet):
    """
    Generate the Fleet sheet dataframe.
    """

    fleet_df = Fleet.copy()

    return fleet_df

In [49]:
def generate_fleetage_sheet(FleetAge):
    """
    Generate the FleetAge sheet dataframe.
    """

    fleetage_df = FleetAge.copy()

    return fleetage_df

In [50]:
def generate_phev_sheet(run_spec_config):
    """
    Generate the PHEV sheet dataframe.
    """

    phev_info = run_spec_config["phev"]

    vehicle_rows = [
        ("LDV (or LDGV for EPA)", "ldv"),
        ("LDT (or LDGT for EPA)", "ldt"),
        ("MDV (or HDGV for EPA)", "mdv"),
        ("HDV (or HDDV for EPA)", "hdv"),
        ("Bus", "bus")
    ]

    phev_df = pd.DataFrame("", index=range(6), columns=range(3))

    # Header row
    phev_df.iloc[0, 0] = ""
    phev_df.iloc[0, 1] = "Fraction (%)"
    phev_df.iloc[0, 2] = "range (km)"

    # Data rows
    for row_index, (label, json_key) in enumerate(vehicle_rows, start=1):
        phev_df.iloc[row_index, 0] = label
        phev_df.iloc[row_index, 1] = float(phev_info[json_key]["phevPercent"])
        phev_df.iloc[row_index, 2] = float(phev_info[json_key]["electricRange"])

    return phev_df

In [51]:
def generate_electric_sheet(run_spec_config):
    """
    Generate the Electric sheet dataframe.
    """

    electrical_grid = run_spec_config["electricalGrid"]

    use_alternate = electrical_grid["useAlternate"]
    factors = electrical_grid["factors"]
    power_shares = electrical_grid["powerShares"]

    # Create empty dataframe with 12 rows and 5 columns: A to E
    electric_df = pd.DataFrame("", index=range(12), columns=range(5))

    # Row 1
    electric_df.iloc[0, 0] = "Technology"
    electric_df.iloc[0, 1] = "Share"

    # Row 2
    electric_df.iloc[1, 0] = "whole-grid emission factors flag"

    if use_alternate:
        electric_df.iloc[1, 1] = 1
        electric_df.iloc[1, 2] = float(factors["co2e"])
        electric_df.iloc[1, 3] = float(factors["so2"])
        electric_df.iloc[1, 4] = float(factors["nox"])
    else:
        electric_df.iloc[1, 1] = 0

        technology_mapping = [
            ("Coal", "coal"),
            ("cogen", "cogeneration"),
            ("CC Gas", "combinedCycle"),
            ("Simple Cycle Gas", "simpleCycleGas"),
            ("Coal to Gas", "coalToGas"),
            ("Hydro", "hydro"),
            ("Wind", "wind"),
            ("Solar", "solar"),
            ("Other", "other")
        ]

        # Rows 4 to 12
        for row_index, (technology_name, json_key) in enumerate(technology_mapping, start=3):
            electric_df.iloc[row_index, 0] = technology_name
            electric_df.iloc[row_index, 1] = float(power_shares[json_key])

    return electric_df

In [52]:
def generate_coldstart_sheet(run_spec_config):
    """
    Generate the ColdStart sheet dataframe.
    """

    coldstart_info = run_spec_config["coldStart"]

    detection_distance = float(coldstart_info["detectionDistance"])
    read_from_emme = coldstart_info["readFromEmme"].strip().lower()

    if read_from_emme == "yes":
        emme_coldstart_flag = 1
    else:
        emme_coldstart_flag = 0

    # Create empty dataframe with 8 rows and 3 columns: A to C
    coldstart_df = pd.DataFrame("", index=range(8), columns=range(3))

    # Row 1
    coldstart_df.iloc[0, 0] = "Cold start distance (km)"
    coldstart_df.iloc[0, 1] = detection_distance

    # Row 2
    coldstart_df.iloc[1, 0] = "Use EMME file coldstart flag"
    coldstart_df.iloc[1, 1] = emme_coldstart_flag

    # Rows 4 to 8 only if readFromEmme is not yes
    if emme_coldstart_flag == 0:

        vehicle_classes = ["LDV", "LDT", "MDV", "HDV", "BUS"]

        for row_index, vehicle_class in enumerate(vehicle_classes, start=3):
            coldstart_df.iloc[row_index, 0] = vehicle_class
            coldstart_df.iloc[row_index, 1] = float(coldstart_info["link"][vehicle_class])
            coldstart_df.iloc[row_index, 2] = float(coldstart_info["zone"][vehicle_class])

    return coldstart_df

In [53]:
def generate_evaporative_sheet(run_spec_config):
    """
    Generate the Evaporative sheet dataframe.
    """

    evaporative_info = run_spec_config["evaporativeEmissions"]
    domain_info = run_spec_config["domainInfo"]

    day_type = domain_info["dayType"].strip().lower()

    if day_type == "weekdays":
        weekday_flag = 1
    else:
        weekday_flag = 0

    evaporative_df = pd.DataFrame({
        0: [
            "Simulation Duration (hours)",
            "Trip per day",
            "Trip average distance (km)",
            "Ratio resting to active vehicles",
            "Weekday flag"
        ],
        1: [
            24,
            3.495,
            float(evaporative_info["avgTripDistance"]),
            float(evaporative_info["restingToActiveRatio"]),
            weekday_flag
        ]
    })

    return evaporative_df

In [54]:
def generate_linkclass_sheet(run_spec_config):
    """
    Generate the LinkClass sheet dataframe.
    """

    link_categories = run_spec_config["linkCategories"]

    # Create empty dataframe with 21 rows and 3 columns: A to C
    linkclass_df = pd.DataFrame("", index=range(21), columns=range(3))

    # Header row
    linkclass_df.iloc[0, 0] = "Link ID"
    linkclass_df.iloc[0, 1] = "Definition Flag"
    linkclass_df.iloc[0, 2] = "Name"

    # Data rows: Link ID 1 to 20
    for row_index, category in enumerate(link_categories, start=1):
        link_id = row_index

        enabled = category["enabled"]
        name = category["name"]

        definition_flag = 1 if enabled else 0

        # If name is empty, use default class name
        if name == "":
            name = f"Class {link_id}"

        linkclass_df.iloc[row_index, 0] = link_id
        linkclass_df.iloc[row_index, 1] = definition_flag
        linkclass_df.iloc[row_index, 2] = name

    return linkclass_df

In [55]:
def export_canmoves_excel(sheets, output_excel_path):
    """
    Export generated CANMOVES sheets into one Excel workbook.
    """

    output_excel_path = Path(output_excel_path)

    output_excel_path.parent.mkdir(parents=True, exist_ok=True)

    with pd.ExcelWriter(output_excel_path) as writer:

        for sheet_name, df in sheets.items():
            df.to_excel(
                writer,
                sheet_name=sheet_name,
                index=False,
                header=False
            )

    return str(output_excel_path)

#Main code

In [57]:
if __name__ == "__main__":

  #Reading JSON files
  json_inputs = read_canmoves_json_files(
      scale_definition_path=CANMOVES_SCALE_DEFINITION_PATH,
      run_spec_config_path=CANMOVES_RUN_SPEC_CONFIG_PATH
  )

  scale_definition = json_inputs["scale_definition"]
  run_spec_config = json_inputs["run_spec_config"]


  #Reading CSV files
  csv_inputs = read_canmoves_csv_files({
      "Fleet": Fleet_CSV_PATH,
      "FleetAge": FleetAge_CSV_PATH
  })
  Fleet = csv_inputs["Fleet"]
  FleetAge = csv_inputs["FleetAge"]



  #Generating sheets
  Traffic = generate_traffic_sheet(scale_definition, run_spec_config, EMME_Path)
  Region = generate_region_sheet(run_spec_config)
  Fleet_sheet = generate_fleet_sheet(Fleet)
  FleetAge_sheet = generate_fleetage_sheet(FleetAge)
  PHEV = generate_phev_sheet(run_spec_config)
  Electric = generate_electric_sheet(run_spec_config)
  ColdStart = generate_coldstart_sheet(run_spec_config)
  Evaporative = generate_evaporative_sheet(run_spec_config)
  LinkClass = generate_linkclass_sheet(run_spec_config)


  #Exporting Excel file
  sheets = {
      "Traffic": Traffic,
      "Region": Region,
      "Fleet": Fleet_sheet,
      "FleetAge": FleetAge_sheet,
      "PHEV": PHEV,
      "Electric": Electric,
      "ColdStart": ColdStart,
      "Evaporative": Evaporative,
      "LinkClass": LinkClass
  }

  saved_file = export_canmoves_excel(
      sheets=sheets,
      output_excel_path=OUTPUT_EXCEL_PATH
  )
